# LLM Counterfactual Fairness Case Study

Hiring language is a classic **counterfactual fairness** setting: hold the job and the candidate constant, change only a demographic marker, and ask whether the model's *text* changes.

This notebook runs fairpipe's `counterfactual_fairness_divergence` probe on **committed Claude Haiku** completions (`claude-haiku-4-5`). Responses are **replayed from cache** — no live API calls, no `ANTHROPIC_API_KEY` required.

The statistic is **not** a classifier rate (demographic parity, equalized odds). For each hiring template, fairpipe pairs the three gender-coded completions (woman / man / nonbinary), extracts lightweight lexical features (sentiment-word mix, refusal phrases, length, token overlap), and records how far those feature vectors diverge. The reported value is the **maximum mean pairwise divergence** across the gender dimension. Token overlap (1 − Jaccard) dominates the score. **0 is not the no-effect baseline:** two long generated texts that differ only by a proper noun already diverge by ~0.19 on this featureization ([BL-012](../docs/fairpipe-technical-backlog.md#bl-012--counterfactual_fairness_divergence-has-no-no-effect-baseline)).

We use the same **`min_group_size` contract as classifier metrics**: groups smaller than the default (**5**) are dropped; if fewer than two groups remain, the metric is **`nan`**. That is a production guard, not a failed run.

**What you will do**

1. **Part A** — n=1 prompt per group (below the guard). Confirm the toolkit *refuses* to report a number.
2. **Part B** — n=9 templates × 3 groups = 27 cached completions. Read a finite divergence, a bootstrap CI, and per-group counts.

**How to run:** select kernel **Python (fairpipe .venv)** (or any kernel whose working directory is this repository). Restart the kernel if imports fail. The first code cell adds the repo root to `sys.path` so the package imports even when Jupyter labeled the kernel `.venv` but launched Homebrew Python 3.12.

## Part A — Guard demonstration (n=1 per group)

The original fixture has **one prompt per group**. The next cell should return **`nan`** and an empty eligible `n_per_group`. That is the correct production default, not a failure of the probe.

In [ ]:
import math
import sys
from pathlib import Path

_here = Path.cwd().resolve()
_root = next(
    (
        p
        for p in (_here, *_here.parents)
        if (p / "fairness_pipeline_dev_toolkit" / "__init__.py").is_file()
        and (p / "pyproject.toml").is_file()
    ),
    None,
)
if _root is None:
    raise RuntimeError(
        f"Could not find the fairpipe repo root from cwd={_here}. "
        "Select kernel 'Python (fairpipe .venv)', restart, and re-run."
    )
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from fairness_pipeline_dev_toolkit.llm_evals import (
    DEFAULT_LLM_MIN_GROUP_SIZE,
    default_recorded_counterfactual_config,
    expanded_recorded_counterfactual_config,
    run_llm_eval,
)

n1_config = default_recorded_counterfactual_config()
blocked = run_llm_eval(n1_config, with_ci=False)
blocked_metric = blocked.metrics["counterfactual_fairness_divergence"]
print(f"Part A — default min_group_size={DEFAULT_LLM_MIN_GROUP_SIZE}")
print(f"Metric at default threshold: {blocked_metric.value}")
print(f"Eligible n_per_group: {blocked_metric.n_per_group}")
assert math.isnan(blocked_metric.value)
print("Guard correctly blocked below-threshold fixture (nan, empty eligible groups).")

### Reading Part A

Expected output: `min_group_size=5`, metric **`nan`**, **`n_per_group: {}`**.

The n=1 fixture still *replays* three cached completions (one each for woman, man, nonbinary). The guard runs **after** that: n=1 is below 5, so no group is eligible and fairpipe will not invent a divergence from a handful of texts.

That is the same rule as `FairnessAnalyzer` on a tiny slice of tabular data. Do **not** pass `allow_small_samples=True` here; that flag is for smoke tests, not for a number you would cite.

Part B uses a **different committed cache** with nine templates so each group reaches n=9 and clears the default threshold.

## Part B — Threshold-clearing fixture (n=9 per group)

Nine hiring templates × three gender groups = **27** live-recorded Haiku responses. Each group has n=9, which clears `min_group_size=5`. There is **no** `allow_small_samples` override.

Bootstrap (`B=200`, seed 42) resamples the **template-level pairwise divergences**: 9 templates × 3 group-pairs = **27** numbers. It does **not** resample tokens inside a single completion.

Run the next cell. It should finish in about a second (cache replay). If it sits for many minutes, the kernel is calling the live Anthropic API instead of the fixture — stop it and confirm `cache_dir` replay (see the intro).

In [ ]:
expanded = expanded_recorded_counterfactual_config()
result = run_llm_eval(expanded, with_ci=True, bootstrap_B=200, random_state=42)
metric = result.metrics["counterfactual_fairness_divergence"]
COUNTERFACTUAL_DIVERGENCE = metric.value
print("Part B — expanded fixture at default min_group_size (no allow_small_samples)")
print(f"Counterfactual fairness divergence: {COUNTERFACTUAL_DIVERGENCE:.4f}")
print(f"95% CI: {metric.ci}")
print(f"n_per_group: {metric.n_per_group}")
print(f"Provider: {expanded.provider} / {expanded.model} (cache replay)")
assert math.isfinite(COUNTERFACTUAL_DIVERGENCE)
assert metric.ci is not None and metric.ci[0] < metric.ci[1]
assert metric.n_per_group == {"woman": 9, "man": 9, "nonbinary": 9}
print("Notebook threshold-clearing check passed.")

### Reading Part B

On this committed cache the cell reports approximately:

| Field | Value | What it means |
|---|---|---|
| Divergence | **0.1956** | Mean matched pairwise feature distance (sentiment, refusal, length, token overlap), then the **max** over the gender dimension. Token overlap (1 − Jaccard) is ~90% of the score. This is **not** “19.6% of candidates were treated unfairly,” and it is **not** a group-effect size. |
| 95% CI | **(0.185, 0.205)** | Percentile bootstrap on the 27 pair values (`B=200`). The interval correctly bounds the statistic and does not include 0. **That does not indicate a group effect.** Zero is not the no-effect baseline for this metric. |
| `n_per_group` | **9 / 9 / 9** | All three groups cleared `min_group_size=5`. |
| Provider | `anthropic` / `claude-haiku-4-5` | Replay of recorded Haiku text, not a live call. |

The reported value measures **lexical divergence between responses**, dominated by token overlap. A within-group control (same asylum template, three same-coded names per group, same model and params; `recorded_within_group_control/`) puts the no-effect baseline at **~0.19**, not 0:

| | mean | min | max | pairs |
|---|---|---|---|---|
| Within-group (same gender coding, different names) | **0.190** | 0.152 | 0.221 | 9 |
| Cross-group (different gender coding) | **0.187** | 0.123 | 0.222 | 27 |

Against that baseline, hiring is 0.196 − 0.190 ≈ 0.006 (inconsistent sign). The humanitarian replay (0.202) is the same. **Neither shows a detectable group effect.** See [BL-012](../docs/fairpipe-technical-backlog.md#bl-012--counterfactual_fairness_divergence-has-no-no-effect-baseline).

Anyone using this metric on their own data will make the same error unless they compare against a within-group baseline rather than 0.

**What this fixture does demonstrate:** the pipeline end to end — recording, replay, guards, CIs, provenance — on real model output.

**Limits (do not over-claim)**

- One model, one temperature (`0.0`), one domain (short hiring notes).
- Features are **lexical**, not embeddings or human ratings — two equally strong recommendations that use different synonyms still score as divergence.
- Gender is the only swapped dimension; names in the templates vary but are not the counterfactual axis.
- Refusal, toxicity, and BBQ probes are **out of scope** here. Toxicity and BBQ shipped demo caches remain illustrative until those BL-009 halves close. Humanitarian refusal is live data but not a disparity finding (15/15 lexical ceiling). Do not treat those metrics as evidence from this notebook.

The same `MetricResult` (`value`, `ci`, `n_per_group`, `caveat`) is what `assert_llm_fairness()`, Markdown reports, and MLflow consume — the same contract as classifier fairness.